<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Air-Quality-_PM-2.5/Air_Quality_PM_2_5_Hybrid_Final_Iterations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install river --quiet

In [ ]:
import pandas as pd
import numpy as np

from river import metrics, compose, preprocessing

print("River imported successfully")

River imported successfully


In [ ]:
path = "/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/final_pm25_dataset.csv"
pm25_df = pd.read_csv(path)

print("Dataset shape:", pm25_df.shape)
pm25_df.head()

Dataset shape: (130003, 16)


,air_quality_PM10,air_quality_Carbon_Monoxide,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,humidity,cloud,visibility_km,longitude,temperature_celsius,condition_text,uv_index,wind_mph,precip_mm,gust_mph,wind_degree,air_quality_PM2.5
0,7.1,198.6,2.5,0.2,58,0,16.0,-120.49,16.1,2,1.0,4.3,0.00,10.3,220,6.3
1,25.3,377.2,3.7,1.4,78,37,10.0,-87.22,23.0,32,1.0,3.8,0.28,7.0,240,19.0
2,28.1,460.6,7.7,7.5,94,50,10.0,-89.20,26.0,23,1.0,2.2,0.30,2.8,182,20.4
3,178.1,2243.0,35.0,19.3,88,100,5.0,-90.53,20.0,19,1.0,13.6,0.09,18.1,190,132.0
4,32.1,307.1,0.3,0.2,89,94,10.0,-88.77,26.0,30,1.0,4.3,0.00,6.5,99,7.7


In [ ]:
target = "air_quality_PM2.5"

In [ ]:
pm25_df["pm25_lag1"] = pm25_df[target].shift(1)
pm25_df["pm25_lag2"] = pm25_df[target].shift(2)
pm25_df["pm25_lag3"] = pm25_df[target].shift(3)
pm25_df["pm25_lag5"] = pm25_df[target].shift(5)
pm25_df["pm25_lag7"] = pm25_df[target].shift(7)

pm25_df["pm25_roll3_mean"] = pm25_df[target].rolling(window=3).mean()
pm25_df["pm25_roll5_mean"] = pm25_df[target].rolling(window=5).mean()
pm25_df["pm25_roll7_mean"] = pm25_df[target].rolling(window=7).mean()

pm25_df["pm25_roll3_std"] = pm25_df[target].rolling(window=3).std()
pm25_df["pm25_roll5_std"] = pm25_df[target].rolling(window=5).std()

if "humidity" in pm25_df.columns:
    pm25_df["humidity_lag1"] = pm25_df["humidity"].shift(1)
    pm25_df["humidity_lag3"] = pm25_df["humidity"].shift(3)

if "temperature_celsius" in pm25_df.columns:
    pm25_df["temp_lag1"] = pm25_df["temperature_celsius"].shift(1)
    pm25_df["temp_lag3"] = pm25_df["temperature_celsius"].shift(3)

if "wind_mph" in pm25_df.columns:
    pm25_df["wind_lag1"] = pm25_df["wind_mph"].shift(1)
    pm25_df["wind_lag3"] = pm25_df["wind_mph"].shift(3)

if "humidity" in pm25_df.columns and "temperature_celsius" in pm25_df.columns:
    pm25_df["humidity_temp_interaction"] = pm25_df["humidity"] * pm25_df["temperature_celsius"]

if "wind_mph" in pm25_df.columns and "air_quality_PM10" in pm25_df.columns:
    pm25_df["wind_pm10_interaction"] = pm25_df["wind_mph"] * pm25_df["air_quality_PM10"]

pm25_df = pm25_df.dropna().reset_index(drop=True)

print("After feature engineering:", pm25_df.shape)
pm25_df.head()

After feature engineering: (129996, 34)


,air_quality_PM10,air_quality_Carbon_Monoxide,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,humidity,cloud,visibility_km,longitude,temperature_celsius,condition_text,...,pm25_roll3_std,pm25_roll5_std,humidity_lag1,humidity_lag3,temp_lag1,temp_lag3,wind_lag1,wind_lag3,humidity_temp_interaction,wind_pm10_interaction
0,48.0,974.7,48.7,16.9,47,5,10.0,-99.13,20.8,2,...,11.741096,51.608216,100.0,89.0,21.0,26.0,2.2,4.3,977.6,321.60
1,20.8,707.6,12.2,12.3,84,84,10.0,-76.75,21.9,41,...,12.510795,11.308758,47.0,80.0,20.8,27.2,6.7,3.6,1839.6,45.76
2,3.9,397.2,8.7,6.6,94,100,10.0,-79.53,26.0,30,...,16.900099,12.467558,84.0,100.0,21.9,21.0,2.2,2.2,2444.0,8.58
3,28.9,2403.3,72.7,52.0,92,76,10.0,-78.50,11.9,41,...,11.984991,12.825443,94.0,47.0,26.0,20.8,2.2,6.7,1094.8,78.03
4,20.2,303.8,12.2,10.6,89,59,10.0,-77.05,16.6,31,...,12.050864,13.370490,92.0,84.0,11.9,21.9,2.7,2.2,1477.4,149.48


In [ ]:
selected_features = [col for col in pm25_df.columns if col != target]

print("Number of features:", len(selected_features))
print(selected_features)

Number of features: 33
['air_quality_PM10', 'air_quality_Carbon_Monoxide', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'humidity', 'cloud', 'visibility_km', 'longitude', 'temperature_celsius', 'condition_text', 'uv_index', 'wind_mph', 'precip_mm', 'gust_mph', 'wind_degree', 'pm25_lag1', 'pm25_lag2', 'pm25_lag3', 'pm25_lag5', 'pm25_lag7', 'pm25_roll3_mean', 'pm25_roll5_mean', 'pm25_roll7_mean', 'pm25_roll3_std', 'pm25_roll5_std', 'humidity_lag1', 'humidity_lag3', 'temp_lag1', 'temp_lag3', 'wind_lag1', 'wind_lag3', 'humidity_temp_interaction', 'wind_pm10_interaction']


In [ ]:
split_index = int(0.8 * len(pm25_df))

train_df = pm25_df.iloc[:split_index].copy()
test_df = pm25_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (103996, 34)
Test shape : (26000, 34)


In [ ]:
n_train = len(train_df)

iter1 = train_df.iloc[:int(0.33 * n_train)].copy()
iter2 = train_df.iloc[int(0.33 * n_train):int(0.66 * n_train)].copy()
iter3 = train_df.iloc[int(0.66 * n_train):].copy()

print("Iteration 1 shape:", iter1.shape)
print("Iteration 2 shape:", iter2.shape)
print("Iteration 3 shape:", iter3.shape)

Iteration 1 shape: (34318, 34)
Iteration 2 shape: (34319, 34)
Iteration 3 shape: (35359, 34)


In [ ]:
try:
    from river import forest

    river_model = forest.ARFRegressor(
        n_models=20,
        max_features="sqrt",
        lambda_value=6,
        seed=42
    )
    model_name = "River ARF Hybrid"

except Exception:
    from river import tree

    river_model = compose.Pipeline(
        preprocessing.StandardScaler(),
        tree.HoeffdingAdaptiveTreeRegressor(
            grace_period=50,
            delta=1e-5,
            leaf_prediction="adaptive"
        )
    )
    model_name = "River HAT Hybrid"

print("Using model:", model_name)

Using model: River ARF Hybrid


In [ ]:
def run_hybrid_iteration(data, model, iteration_name, selected_features, target, warmup=False):
    mse = metrics.MSE()
    rmse = metrics.RMSE()
    mae = metrics.MAE()
    r2 = metrics.R2()

    y_true_all = []
    y_pred_all = []

    first_part = int(0.1 * len(data)) if warmup else 0

    for i, (_, row) in enumerate(data.iterrows()):
        x = row[selected_features].to_dict()
        y = row[target]

        if i < first_part:
            model.learn_one(x, y)
            continue

        y_pred = model.predict_one(x)
        if y_pred is None:
            y_pred = 0.0

        y_true_all.append(y)
        y_pred_all.append(y_pred)

        mse.update(y, y_pred)
        rmse.update(y, y_pred)
        mae.update(y, y_pred)
        r2.update(y, y_pred)

        model.learn_one(x, y)

    print(f"\n{iteration_name}")
    print("MSE :", round(mse.get(), 4))
    print("RMSE:", round(rmse.get(), 4))
    print("MAE :", round(mae.get(), 4))
    print("R2  :", round(r2.get(), 4))
    print("Accuracy (%):", round(r2.get() * 100, 2))

    return model, mse.get(), rmse.get(), mae.get(), r2.get(), y_true_all, y_pred_all

In [ ]:
river_results = []

river_model, mse1, rmse1, mae1, r21, y_true1, y_pred1 = run_hybrid_iteration(
    iter1, river_model, "Iteration 1", selected_features, target, warmup=True
)

river_results.append([
    "Iteration 1", model_name, mse1, rmse1, mae1, r21, r21 * 100
])


Iteration 1
MSE : 448.2657
RMSE: 21.1723
MAE : 5.2554
R2  : 0.7782
Accuracy (%): 77.82


In [ ]:
river_model, mse2, rmse2, mae2, r22, y_true2, y_pred2 = run_hybrid_iteration(
    iter2, river_model, "Iteration 2", selected_features, target, warmup=False
)

river_results.append([
    "Iteration 2", model_name, mse2, rmse2, mae2, r22, r22 * 100
])


Iteration 2
MSE : 365.6862
RMSE: 19.1229
MAE : 7.4692
R2  : 0.7563
Accuracy (%): 75.63


In [ ]:
river_model, mse3, rmse3, mae3, r23, y_true3, y_pred3 = run_hybrid_iteration(
    iter3, river_model, "Iteration 3", selected_features, target, warmup=False
)

river_results.append([
    "Iteration 3", model_name, mse3, rmse3, mae3, r23, r23 * 100
])


Iteration 3
MSE : 177.7685
RMSE: 13.333
MAE : 5.1429
R2  : 0.8271
Accuracy (%): 82.71


# **Training Results**

In [ ]:
river_results_df = pd.DataFrame(
    river_results,
    columns=["Iteration", "Model", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]
).round(3)

river_results_df

,Iteration,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Iteration 1,River ARF Hybrid,448.266,21.172,5.255,0.778,77.825
1,Iteration 2,River ARF Hybrid,365.686,19.123,7.469,0.756,75.634
2,Iteration 3,River ARF Hybrid,177.769,13.333,5.143,0.827,82.708


# **Test Results**

In [ ]:
test_mse = metrics.MSE()
test_rmse = metrics.RMSE()
test_mae = metrics.MAE()
test_r2 = metrics.R2()

test_true = []
test_pred = []

for _, row in test_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    y_hat = river_model.predict_one(x)
    if y_hat is None:
        y_hat = 0.0

    test_true.append(y)
    test_pred.append(y_hat)

    test_mse.update(y, y_hat)
    test_rmse.update(y, y_hat)
    test_mae.update(y, y_hat)
    test_r2.update(y, y_hat)

print("\nFinal Test Performance")
print("MSE :", round(test_mse.get(), 4))
print("RMSE:", round(test_rmse.get(), 4))
print("MAE :", round(test_mae.get(), 4))
print("R2  :", round(test_r2.get(), 4))
print("Accuracy (%):", round(test_r2.get() * 100, 2))


Final Test Performance
MSE : 94.7493
RMSE: 9.7339
MAE : 3.9331
R2  : 0.8417
Accuracy (%): 84.17


In [ ]:
river_final_test_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(test_mse.get(), 3),
    "RMSE": round(test_rmse.get(), 3),
    "MAE": round(test_mae.get(), 3),
    "R2": round(test_r2.get(), 3),
    "Accuracy (%)": round(test_r2.get() * 100, 3)
}])

river_final_test_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River ARF Hybrid,94.749,9.734,3.933,0.842,84.175


In [ ]:
from google.colab import files

river_results_df.to_csv("river_hybrid_pm25_iteration_results.csv", index=False)
river_final_test_df.to_csv("river_hybrid_pm25_final_test_results.csv", index=False)

files.download("river_hybrid_pm25_iteration_results.csv")
files.download("river_hybrid_pm25_final_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>